# 4. Final model: Random Forest (max_depth = 10)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, f1_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)

RANDOM_STATE = 42
DATA_PATH = Path("../data")
FIGURES_PATH = Path("../figures")

## Load train and test sets (first and only use of the test set)

In [ ]:
train_data = pd.read_csv(DATA_PATH / "train_processed.csv")
test_data = pd.read_csv(DATA_PATH / "test_processed.csv")

X_train = train_data.drop(columns="y")
y_train = train_data["y"]
X_test = test_data.drop(columns="y")
y_test = test_data["y"]
print(X_train.shape, X_test.shape)

## Final pipeline: same preprocessing + best hyperparameter from notebook 3

In [ ]:
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()
numerical_features = X_train.select_dtypes(include="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("numerical", StandardScaler(), numerical_features),
    ("categorical", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                  min_frequency=100,
                                  sparse_output=False), categorical_features),
])

final_model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(max_depth=10, class_weight="balanced",
                                          random_state=RANDOM_STATE)),
])

## Training on the full train set

In [ ]:
final_model.fit(X_train, y_train)

## Evaluation on the test set

In [ ]:
test_probabilities = final_model.predict_proba(X_test)[:, 1]
test_predictions = final_model.predict(X_test)

test_roc_auc = roc_auc_score(y_test, test_probabilities)
test_f1 = f1_score(y_test, test_predictions)
print(f"Test ROC-AUC: {test_roc_auc:.3f}")
print(f"Test F1:      {test_f1:.3f}")

## Expected performance on new data: cross-validation (notebook 3) vs. test

In [ ]:
expected_performance = pd.DataFrame({
    "cross_validation (train)": [0.795, 0.476],
    "test": [test_roc_auc, test_f1],
}, index=["roc_auc", "f1"])
expected_performance.round(3)

## Precision and recall per class

In [ ]:
print(classification_report(y_test, test_predictions, target_names=["no", "yes"], digits=3))

## Confusion matrix (test)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions,
                                        display_labels=["no", "yes"], cmap="Blues")
plt.title("Confusion matrix - Random Forest (test)")
plt.savefig(FIGURES_PATH / "11_confusion_matrix_test.png", bbox_inches="tight")
plt.show()

## ROC curve (test)

In [ ]:
RocCurveDisplay.from_predictions(y_test, test_probabilities, name="Random Forest")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", label="random classifier")
plt.legend()
plt.title("ROC curve (test)")
plt.savefig(FIGURES_PATH / "12_roc_curve_test.png", bbox_inches="tight")
plt.show()

## Feature importance (top 15)

In [ ]:
feature_names = final_model.named_steps["preprocessing"].get_feature_names_out()
feature_importances = pd.Series(
    final_model.named_steps["classifier"].feature_importances_,
    index=feature_names,
).sort_values()

feature_importances.tail(15).plot(kind="barh", figsize=(8, 6))
plt.xlabel("Mean impurity decrease")
plt.title("Random Forest - feature importance")
plt.savefig(FIGURES_PATH / "13_feature_importance.png", bbox_inches="tight")
plt.show()